### Setting

In [ ]:
import os
import pandas as pd

import torch
import torch.nn as nn

import gensim
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

import re
import numpy as np

from sklearn.preprocessing import LabelEncoder

import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

import torch.nn.functional as F

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available!")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU.")

GPU not available, using CPU.


### Data Load

* Each playlist has 50 songs
* 2151 playlists has been used (107550 songs in total)
* train: 1700 playlists (85000 songs) test:451 playlists

In [ ]:
dataset_path = '/Users/seoyeonpark/Desktop/DATA5011/Data_Creation/datasetCreated/100k_preprocessed_with_padding.parquet'

data = pd.read_parquet(dataset_path)

# If necessary, you can deserialize the embeddings back into numpy arrays
data['track_embeddings'] = data['track_embeddings'].apply(np.array)
data['album_embeddings'] = data['album_embeddings'].apply(np.array)

data

,pid,track_name,album_name,artist_name,danceability,energy,key,loudness,mode,speechiness,...,song_count,lemmatized_track_name,lemmatized_album_name,track_sentences,album_sentences,track_id,artist_id,audio_features,track_embeddings,album_embeddings
0,0,A Thousand Miles,Be Not Nobody,Vanessa Carlton,0.5590,0.845,11.0,-3.871,1.0,0.0379,...,51,thousand mile,nobody,"[thousand, mile]",[nobody],766,12663,"[0.559, 0.845, 11.0, -3.871, 1.0, 0.0379, 0.22...","[[0.0791015625, -0.060302734375, -0.0157470703...","[[0.1708984375, -0.1669921875, 0.0927734375, 0..."
1,0,Somebody To Love,My Worlds,Justin Bieber,0.7190,0.839,5.0,-5.235,1.0,0.0257,...,51,somebody love,world,"[somebody, love]",[world],31876,6099,"[0.719, 0.839, 5.0, -5.235, 1.0, 0.0257, 0.002...","[[0.302734375, -0.11474609375, 0.05712890625, ...","[[-0.06396484375, 0.068359375, 0.224609375, 0...."
2,0,How Do You Sleep? - Featuring Ludacris,Departure - Recharged,Jesse McCartney,0.7470,0.878,8.0,-3.533,1.0,0.0282,...,51,sleep feature ludacris,departure recharge,"[sleep, feature, ludacris]","[departure, recharge]",15643,5600,"[0.747, 0.878, 8.0, -3.533, 1.0, 0.0282, 0.343...","[[-0.07177734375, -0.10009765625, -0.120117187...","[[0.049560546875, 0.0106201171875, -0.21972656..."
3,0,Beautiful Soul,Beautiful Soul,Jesse McCartney,0.6600,0.666,9.0,-4.342,1.0,0.0472,...,51,beautiful soul,beautiful soul,"[beautiful, soul]","[beautiful, soul]",3426,5600,"[0.66, 0.666, 9.0, -4.342, 1.0, 0.0472, 0.0759...","[[-0.018310546875, 0.0556640625, -0.0115356445...","[[-0.018310546875, 0.0556640625, -0.0115356445..."
4,0,Year 3000,Jonas Brothers,Jonas Brothers,0.6590,0.869,11.0,-5.858,1.0,0.0460,...,51,year,jonas brother,[year],"[jonas, brother]",40712,5890,"[0.659, 0.869, 11.0, -5.858, 1.0, 0.046, 0.003...","[[0.061767578125, 0.2578125, 0.003677368164062...","[[-0.1259765625, 0.0400390625, 0.12451171875, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107545,101462,Meguru Kisetsu,Ghibli's World with Piano Vol. 2,Kanako Oba,0.5310,0.137,9.0,-24.634,0.0,0.0587,...,75,meguru kisetsu,ghiblis world piano vol,"[meguru, kisetsu]","[ghiblis, world, piano, vol]",22723,6184,"[0.531, 0.137, 9.0, -24.634, 0.0, 0.0587, 0.99...","[[-0.5714038014411926, -0.32191339135169983, -...","[[-0.7683327794075012, -0.93965083360672, 0.84..."
107546,101462,Rey's Theme,Star Wars: The Force Awakens,John Williams,0.2340,0.139,9.0,-16.854,0.0,0.0342,...,75,reys theme,star war force awakens,"[reys, theme]","[star, war, force, awakens]",28833,5828,"[0.234, 0.139, 9.0, -16.854, 0.0, 0.0342, 0.97...","[[0.004046235699206591, -0.3215627074241638, 0...","[[0.1640625, 0.1884765625, 0.1416015625, -0.02..."
107547,101462,Story of My Life,Wonders,The Piano Guys,0.3180,0.445,3.0,-9.588,1.0,0.0349,...,75,story life,wonder,"[story, life]",[wonder],32883,11860,"[0.318, 0.445, 3.0, -9.588, 1.0, 0.0349, 0.986...","[[0.17578125, 0.0245361328125, -0.1689453125, ...","[[0.058837890625, -0.0301513671875, 0.02038574..."
107548,101462,Fullmetal Alchemist: Brotherhood - Lapis Philo...,Piano Solo Works: Anime Themes - Volume I,Michael Tai,0.5480,0.168,4.0,-14.317,0.0,0.0896,...,75,fullmetal alchemist brotherhood lapis philosop...,piano solo work anime theme volume,"[fullmetal, alchemist, brotherhood, lapis, phi...","[piano, solo, work, anime, theme, volume]",12480,7930,"[0.548, 0.168, 4.0, -14.317, 0.0, 0.0896, 0.99...","[[-1.2319812774658203, -1.2000256776809692, -0...","[[0.1591796875, -0.17578125, -0.1044921875, -0..."


In [ ]:
unique_pids = data['pid'].unique()
pid_mapping = {pid: new_id for new_id, pid in enumerate(unique_pids)}
data['new_pid'] = data['pid'].map(pid_mapping)
data

,pid,track_name,album_name,artist_name,danceability,energy,key,loudness,mode,speechiness,...,lemmatized_track_name,lemmatized_album_name,track_sentences,album_sentences,track_id,artist_id,audio_features,track_embeddings,album_embeddings,new_pid
0,0,A Thousand Miles,Be Not Nobody,Vanessa Carlton,0.5590,0.845,11.0,-3.871,1.0,0.0379,...,thousand mile,nobody,"[thousand, mile]",[nobody],766,12663,"[0.559, 0.845, 11.0, -3.871, 1.0, 0.0379, 0.22...","[[0.0791015625, -0.060302734375, -0.0157470703...","[[0.1708984375, -0.1669921875, 0.0927734375, 0...",0
1,0,Somebody To Love,My Worlds,Justin Bieber,0.7190,0.839,5.0,-5.235,1.0,0.0257,...,somebody love,world,"[somebody, love]",[world],31876,6099,"[0.719, 0.839, 5.0, -5.235, 1.0, 0.0257, 0.002...","[[0.302734375, -0.11474609375, 0.05712890625, ...","[[-0.06396484375, 0.068359375, 0.224609375, 0....",0
2,0,How Do You Sleep? - Featuring Ludacris,Departure - Recharged,Jesse McCartney,0.7470,0.878,8.0,-3.533,1.0,0.0282,...,sleep feature ludacris,departure recharge,"[sleep, feature, ludacris]","[departure, recharge]",15643,5600,"[0.747, 0.878, 8.0, -3.533, 1.0, 0.0282, 0.343...","[[-0.07177734375, -0.10009765625, -0.120117187...","[[0.049560546875, 0.0106201171875, -0.21972656...",0
3,0,Beautiful Soul,Beautiful Soul,Jesse McCartney,0.6600,0.666,9.0,-4.342,1.0,0.0472,...,beautiful soul,beautiful soul,"[beautiful, soul]","[beautiful, soul]",3426,5600,"[0.66, 0.666, 9.0, -4.342, 1.0, 0.0472, 0.0759...","[[-0.018310546875, 0.0556640625, -0.0115356445...","[[-0.018310546875, 0.0556640625, -0.0115356445...",0
4,0,Year 3000,Jonas Brothers,Jonas Brothers,0.6590,0.869,11.0,-5.858,1.0,0.0460,...,year,jonas brother,[year],"[jonas, brother]",40712,5890,"[0.659, 0.869, 11.0, -5.858, 1.0, 0.046, 0.003...","[[0.061767578125, 0.2578125, 0.003677368164062...","[[-0.1259765625, 0.0400390625, 0.12451171875, ...",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107545,101462,Meguru Kisetsu,Ghibli's World with Piano Vol. 2,Kanako Oba,0.5310,0.137,9.0,-24.634,0.0,0.0587,...,meguru kisetsu,ghiblis world piano vol,"[meguru, kisetsu]","[ghiblis, world, piano, vol]",22723,6184,"[0.531, 0.137, 9.0, -24.634, 0.0, 0.0587, 0.99...","[[-0.5714038014411926, -0.32191339135169983, -...","[[-0.7683327794075012, -0.93965083360672, 0.84...",2150
107546,101462,Rey's Theme,Star Wars: The Force Awakens,John Williams,0.2340,0.139,9.0,-16.854,0.0,0.0342,...,reys theme,star war force awakens,"[reys, theme]","[star, war, force, awakens]",28833,5828,"[0.234, 0.139, 9.0, -16.854, 0.0, 0.0342, 0.97...","[[0.004046235699206591, -0.3215627074241638, 0...","[[0.1640625, 0.1884765625, 0.1416015625, -0.02...",2150
107547,101462,Story of My Life,Wonders,The Piano Guys,0.3180,0.445,3.0,-9.588,1.0,0.0349,...,story life,wonder,"[story, life]",[wonder],32883,11860,"[0.318, 0.445, 3.0, -9.588, 1.0, 0.0349, 0.986...","[[0.17578125, 0.0245361328125, -0.1689453125, ...","[[0.058837890625, -0.0301513671875, 0.02038574...",2150
107548,101462,Fullmetal Alchemist: Brotherhood - Lapis Philo...,Piano Solo Works: Anime Themes - Volume I,Michael Tai,0.5480,0.168,4.0,-14.317,0.0,0.0896,...,fullmetal alchemist brotherhood lapis philosop...,piano solo work anime theme volume,"[fullmetal, alchemist, brotherhood, lapis, phi...","[piano, solo, work, anime, theme, volume]",12480,7930,"[0.548, 0.168, 4.0, -14.317, 0.0, 0.0896, 0.99...","[[-1.2319812774658203, -1.2000256776809692, -0...","[[0.1591796875, -0.17578125, -0.1044921875, -0...",2150


# Data Preparation


### Train Test Split

In [ ]:
train = data.iloc[:95000,:]
#valid = data.iloc[85000:95000,:]
test = data.iloc[95000:,:]

print(train.shape)
#print(valid.shape)
print(test.shape)

(95000, 29)
(12550, 29)


In [ ]:
train.groupby('pid')['track_name'].transform('count').describe()

count    95000.0
mean        50.0
std          0.0
min         50.0
25%         50.0
50%         50.0
75%         50.0
max         50.0
Name: track_name, dtype: float64

In [ ]:
test.groupby('pid')['track_name'].transform('count').describe()

count    12550.0
mean        50.0
std          0.0
min         50.0
25%         50.0
50%         50.0
75%         50.0
max         50.0
Name: track_name, dtype: float64

### Creating I+ and I- only for training data

In [ ]:
def create_i_plus_i_minus_balanced(train_data, playlist_length=50):
    # I+ (Positive interactions): All playlist-track pairs in train_data
    positive_interactions = train_data[['pid', 'track_id']].copy()

    # I- (Negative interactions): Random tracks not on the playlist, same count as I+
    negative_interactions = []
    all_tracks = set(data['track_id'].unique())  # Set of all track IDs in the dataset

    for pid in train_data['pid'].unique():
        # Get all positive tracks for the current playlist (I+)
        positive_tracks = set(train_data[train_data['pid'] == pid]['track_id'])
        num_positive_samples = playlist_length

        # Only proceed if there are positive samples
        if num_positive_samples > 0:
            # Tracks that are not in the playlist (I-)=
            negative_tracks_candidates = list(all_tracks - positive_tracks)

            # Sample an equal number of negative tracks
            sampled_negative_tracks = np.random.choice(negative_tracks_candidates, size=num_positive_samples, replace=False)
            #print(len(positive_tracks),len(sampled_negative_tracks))

            # Append negative interactions for the current playlist
            for track_id in sampled_negative_tracks:
                negative_interactions.append([pid, track_id])
            #############print(pid, len(positive_tracks),len(sampled_negative_tracks), len(negative_interactions))

    # Convert the negative interactions to a DataFrame
    negative_interactions = pd.DataFrame(negative_interactions, columns=['pid', 'track_id'])

    # Return both positive (I+) and negative (I-) interactions
    return positive_interactions, negative_interactions


In [ ]:
# Call the function to generate balanced positive and negative interactions
positive_interactions, negative_interactions = create_i_plus_i_minus_balanced(train)

# Check if the lengths of positive and negative interactions match
print(f"Number of positive interactions: {len(positive_interactions)}")
print(f"Number of negative interactions: {len(negative_interactions)}")

Number of positive interactions: 95000
Number of negative interactions: 95000


### Combine I+ / I- with meta embedding and audio feature sets

In [ ]:
# Step 1: Add an index column to the original dataframe to keep track of the order
positive_interactions['original_index'] = positive_interactions.index

train_unique = train.drop_duplicates(subset=['track_id'], keep='first')

# Step 2: Merge the dataframes
I_plus = pd.merge(positive_interactions, train_unique, on='track_id', how='left')

# Step 3: Sort the merged dataframe by the original index to preserve the order
I_plus = I_plus.sort_values(by='original_index').reset_index(drop=True)

# Step 4: Drop the index column if no longer needed
I_plus = I_plus.drop(columns=['original_index','pid_y'])
I_plus.rename(columns={'pid_x': 'pid'}, inplace=True)

I_plus['new_pid'] = I_plus['pid'].map(pid_mapping)

In [ ]:
# Step 1: Add an index column to the original dataframe to keep track of the order
negative_interactions['original_index'] = negative_interactions.index

data_unique = data.drop_duplicates(subset=['track_id'], keep='first')

# Step 2: Merge the dataframes
I_minus = pd.merge(negative_interactions, data_unique, on='track_id', how='left')

# Step 3: Sort the merged dataframe by the original index to preserve the order
I_minus = I_minus.sort_values(by='original_index').reset_index(drop=True)

# Step 4: Drop the index column if no longer needed
I_minus = I_minus.drop(columns=['original_index','pid_y'])
I_minus.rename(columns={'pid_x': 'pid'}, inplace=True)

I_minus['new_pid'] = I_minus['pid'].map(pid_mapping)

In [ ]:
I_plus_train = I_plus[:85000]
I_plus_val = I_plus[85000:]

I_minus_train = I_minus[:85000]
I_minus_val = I_minus[85000:]

#### Save I_plus, I_minus, test

In [ ]:
I_plus_train.to_parquet('I_plus_train_100k_padding.parquet')
I_minus_train.to_parquet('I_minus_train_100k_padding.parquet')

In [ ]:
I_plus_val.to_parquet('I_plus_val_100k_padding.parquet')
I_minus_val.to_parquet('I_minus_val_100k_padding.parquet')

In [ ]:
test.to_parquet('test_100k_padding.parquet')
#test